In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from tensorflow import keras
from tensorflow.keras import layers

df = pd.read_csv('/content/bengaluru_house_prices.csv')
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [15]:
def convert_sqft_to_num(x):
    try:
        tokens = str(x).split('-')
        if len(tokens) == 2:
            return (float(tokens[0]) + float(tokens[1])) / 2
        return float(x)
    except:
        return None

In [17]:
df['total_sqft'] = df['total_sqft'].apply(convert_sqft_to_num)
df['bhk'] = df['size'].str.extract('(\d+)').astype(float)
df = df.dropna(subset=['location', 'total_sqft', 'bhk', 'price'])
df['bath'] = df['bath'].fillna(df['bath'].median())
df['balcony'] = df['balcony'].fillna(df['balcony'].median())
df = df[~(df['total_sqft'] / df['bhk'] < 300)]
df = df[~(df['bath'] > df['bhk'] + 2)]

df['price_per_sqft'] = df['price'] * 100000 / df['total_sqft']


<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_2910/1143849237.py:2: SyntaxWarning: invalid escape sequence '\d'
  df['bhk'] = df['size'].str.extract('(\d+)').astype(float)


In [18]:
def remove_pps_outliers(data):
    df_out = pd.DataFrame()
    for key, subdf in data.groupby('location'):
        m = np.mean(subdf['price_per_sqft'])
        st = np.std(subdf['price_per_sqft'])
        reduced = subdf[(subdf['price_per_sqft'] > (m - st)) & (subdf['price_per_sqft'] <= (m + st))]
        df_out = pd.concat([df_out, reduced], ignore_index=True)
    return df_out

df = remove_pps_outliers(df)

In [19]:
df['location'] = df['location'].apply(lambda x: x.strip())
location_counts = df['location'].value_counts()
locations_less_than_10 = location_counts[location_counts <= 10]
df['location'] = df['location'].apply(lambda x: 'other' if x in locations_less_than_10 else x)
df['bath_per_bhk'] = df['bath'] / df['bhk']
features = ['location', 'area_type', 'total_sqft', 'bath', 'balcony', 'bhk', 'bath_per_bhk']
X = df[features]
y = df['price']

num_cols = ['total_sqft', 'bath', 'balcony', 'bhk', 'bath_per_bhk']
cat_cols = ['location', 'area_type']

In [20]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_prep = preprocessor.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_prep, y, test_size=0.2, random_state=42)

In [21]:
X_prep = preprocessor.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_prep, y, test_size=0.2, random_state=42)

model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.15),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.15),

    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

In [25]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.005),
    loss='mse',
    metrics=['mae']
)


early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True)
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=0.0001)


history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=200,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

Epoch 1/200
233/233 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 5157.9624 - mae: 29.4225 - val_loss: 4342.9624 - val_mae: 27.9363 - learning_rate: 0.0050
Epoch 2/200
233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 4309.9058 - mae: 27.7428 - val_loss: 3052.4028 - val_mae: 26.2270 - learning_rate: 0.0050
Epoch 3/200
233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 3865.3132 - mae: 27.1906 - val_loss: 3923.4917 - val_mae: 24.3703 - learning_rate: 0.0050
Epoch 4/200
233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 3497.7046 - mae: 26.0956 - val_loss: 3564.8176 - val_mae: 31.0767 - learning_rate: 0.0050
Epoch 5/200
233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 3318.4314 - mae: 26.2296 - val_loss: 2659.2561 - val_mae: 24.8915 - learning_rate: 0.0050
Epoch 6/200
233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 4132.6016 - mae: 26.4878 - val_loss: 3052.2371 - val_mae: 24.7238 - learning_rate: 0.0050
Epoch 7/200
233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 3323.0828 - mae: 24.6935 - val_lo

In [26]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

y_pred = model.predict(X_test).flatten()
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))


print ("Results")
print(f"R2 Score : {r2:.4f}")
print(f"MAE      : {mae:.2f} Lakhs")
print(f"RMSE     : {rmse:.2f} Lakhs")

59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Results
R2 Score : 0.8250
MAE      : 19.16 Lakhs
RMSE     : 41.99 Lakhs
